In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd



# Generate conditions for symmetric distractor test.

Test in style of Kidd / Marrone et al. (2008) / Srinivasan et al. (2016) experiments 

Target at 0 aziuth    
Distractors symmetrically presented at +/- 0, 5, 10, 20, and 40 degrees azimuth    
All signals at 0 elevation     
Left and right distractor level summed and set to 60dB SPL
- equal power at left and right for model first pass
- human experiments roved L/R balance slightly

Target set against distractor at desired SNR 

## This Notebook
Generate location x snr conditions to get model performance and thresholds

In [2]:
room_ir_df = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl')
elevations = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_elev.unique() # np.arange(-20, 41, 10)
azimuths = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_azim.unique() # np.arange(-20, 41, 10)
# remove 0 elevation
print(azimuths)
print(elevations)


[  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175
 180 185 190 195 200 205 210 215 220 225 230 235 240 245 250 255 260 265
 270 275 280 285 290 295 300 305 310 315 320 325 330 335 340 345 350 355]
[-20 -10   0  10  20  30  40]


### Get range of conditions that match human tests (ours and published)

In [5]:
np.linspace(6,-12,num=7)

array([  6.,   3.,   0.,  -3.,  -6.,  -9., -12.])

In [6]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
snrs = np.linspace(6,-12,num=7)
print(snrs)
# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': snr} for ix, (dist_az, snr) in
              enumerate(itertools.product(*[distractor_azims, snrs]))} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

[  6.   3.   0.  -3.  -6.  -9. -12.]
56


In [12]:
# all_conds
# all conditions 
with open('binaural_test_manifests/symmetric_distractor_conditions_neg_12_to_6_dBSNR.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

In [9]:
# all_conds
# all conditions 
with open('binaural_test_manifests/symmetric_distractor_conditions_neg_12_to_6_dBSNR.pkl', 'rb') as f:
    cond_dict = pickle.load(f)

In [8]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
# snrs = np.linspace(3,-15,num=7)
# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': 6} for ix, dist_az in
              enumerate(distractor_azims, )} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

8


### Test manifest combining conditions used in 11/29/2023 test with all conditions for 0 elev confusion matrix 

In [46]:
# First add conditions for confusion matrix 
target_azims = np.arange(0, 91, 10)
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])
cond_dict = {ix:[(target_az, 0), (dist_az, 0)] for ix, (target_az, dist_az) in
              enumerate(itertools.product(*[target_azims, azimuths]))} 

n_0_elev_conds = len(cond_dict)
print(n_0_elev_conds)
# Now add conditions for all other analyses 
target_azims = np.array([0, 10, 20, 30, 40, 60, 90])
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])


all_other_cond_dict = {ix + n_0_elev_conds:[(target_az, target_elev), (dist_az, dist_elev)] for ix, (target_az, target_elev, dist_az, dist_elev) in
              enumerate(itertools.product(*[target_azims, wanted_elevations, azimuths, wanted_elevations]))} 
print(len(all_other_cond_dict))

# combine dicts 
all_cond_dict = {**cond_dict, **all_other_cond_dict}
print(len(all_cond_dict))

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

361
147
508


In [29]:
# all_conds
# all conditions 
with open('all_9253_room_conds_12_13_2023.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

## Add conditions for full confusion matrix at 0 elevation 

In [4]:
azimuths

array([  0,  10,  20,  30,  40,  50,  60,  70,  80,  90, 350, 340, 330,
       320, 310, 300, 290, 280, 270])

In [5]:
# target_azims = np.array([0, 20, 30, 40, 60, 90])
target_azims = np.array([10, 50, 70, 80])
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])

cond_dict = {ix:[(target_az, 0), (dist_az, 0)] for ix, (target_az, dist_az) in
              enumerate(itertools.product(*[target_azims, azimuths]))} 
len(cond_dict)


133

In [6]:
# all_conds
# all conditions 
with open('compliment_11_29_2023_locs_for_0_elev.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)

### For tests run 11/29/2023 and after

In [ ]:
target_azims = np.array([0, 20, 30, 40, 60, 90])
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])

cond_dict = {ix:[(target_az, target_elev), (dist_az, dist_elev)] for ix,(target_az, target_elev, dist_az, dist_elev) in
              enumerate(itertools.product(*[target_azims, wanted_elevations, azimuths, wanted_elevations]))} 
len(cond_dict)


In [25]:
# all_conds
# all conditions 
with open('expanded_all_azim_chosen_elev_az_11_29_2023.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)

#### For test ran approx 11/14/2023

In [64]:
all_pairs = []
for target_az, dist_az in itertools.product(*[azimuths, azimuths]): 
    all_pairs.append([(target_az, 60), (dist_az, 0)])
    all_pairs.append([(target_az, 0), (dist_az, 60)])
    all_pairs.append([(target_az, 40), (dist_az, -10)])
    all_pairs.append([(target_az, -10), (dist_az, 40)])

all_pairs_dict = {i: pair for i, pair in enumerate(all_pairs)}

In [66]:
len(all_pairs_dict)

1444

In [69]:
all_pairs_dict[1443]

[(270, -10), (270, 40)]

In [67]:
# all_conds
# all conditions 
with open('expanded_elev_v01_11_14_2023.pkl', 'wb') as f:
    pickle.dump(all_pairs_dict, f)

#### For test ran approx 10/29/2023

In [67]:

loc_conds_t_40_az_d_0_az = {ix:[(target_az, 40), (dist_az, 0)] for ix, (target_az, dist_az) in enumerate(itertools.product(*[azimuths, azimuths]))}
loc_conds_t_0_az_d_40_az = {ix:[(target_az, 0), (dist_az, 40)] for ix, (target_az, dist_az) in enumerate(itertools.product(*[azimuths, azimuths]))}


In [68]:
# write condition dicts to pickle files
with open('loc_conds_t_40_az_d_0_az.pkl', 'wb') as f:
    pickle.dump(loc_conds_t_40_az_d_0_az, f)

with open('loc_conds_t_0_az_d_40_az.pkl', 'wb') as f:
    pickle.dump(loc_conds_t_0_az_d_40_az, f)

In [84]:
# new dictionary with values of both dicts with keys indexed from 0 to 720
all_conds = {}
for i in range(722):
    if i <= 360:
        all_conds[i] = loc_conds_t_40_az_d_0_az[i]
    else:
        all_conds[i] = loc_conds_t_0_az_d_40_az[i-361]



In [89]:
# all_conds
# all conditions 
with open('target_40_elev_dist_0_elev_and_target_0_elev_dist_40_elev.pkl', 'wb') as f:
    pickle.dump(all_conds, f)

In [91]:
all_conds[721]

[(270, 0), (270, 40)]